In [2]:
"""
Analysis 4: Leave-One-Dog-Out LDA Decoding
Neuro 120 Final Project — Kohl & Sliz

For each of the 19 overlapping 50ms time windows:
  - Species decoding: classify human vs. dog vocalizations from ERP amplitude
  - Valence decoding: classify positive vs. neutral vocalizations from ERP amplitude

each dog contributes two observations per decoding task (one per condition label),
with ERP amplitude averaged across Fz and Cz and across the other factor.
LOO leaves out both observations from the held-out dog in each fold.
Significance tested against 50% chance via permutation test (n=1000).
"""

import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneGroupOut
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# load data
df = pd.read_csv(
    'data_17dogs.csv',
    sep=';',
    decimal=','
)

TIME_WINDOWS = [
    '0-100', '50-150', '100-200', '150-250', '200-300',
    '250-350', '300-400', '350-450', '400-500', '450-550',
    '500-600', '550-650', '600-700', '650-750', '700-800',
    '750-850', '800-900', '850-950', '900-1000'
]
N_WINDOWS = len(TIME_WINDOWS)
N_PERMS = 1000
CHANCE = 0.5
ALPHA = 0.05

# helper: run LOO-LDA and permutation test for one decoding task

def run_loo_lda(feature_matrix, labels, groups, n_perms=N_PERMS):

    logo = LeaveOneGroupOut()
    lda = LinearDiscriminantAnalysis()
    n_obs = len(labels)

    # observed accuracy
    acc = np.zeros(N_WINDOWS)
    for w, window in enumerate(TIME_WINDOWS):
        X = feature_matrix[:, w].reshape(-1, 1)
        correct = 0
        total = 0
        for train_idx, test_idx in logo.split(X, labels, groups):
            lda.fit(X[train_idx], labels[train_idx])
            preds = lda.predict(X[test_idx])
            correct += np.sum(preds == labels[test_idx])
            total += len(test_idx)
        acc[w] = correct / total

    # permutation test
    null_dist = np.zeros((n_perms, N_WINDOWS))
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(42)

    for p in range(n_perms):
        # permute labels within each dog to preserve group structure
        perm_labels = labels.copy()
        for g in unique_groups:
            idx = np.where(groups == g)[0]
            perm_labels[idx] = rng.permutation(labels[idx])

        for w, window in enumerate(TIME_WINDOWS):
            X = feature_matrix[:, w].reshape(-1, 1)
            correct = 0
            total = 0
            for train_idx, test_idx in logo.split(X, perm_labels, groups):
                lda.fit(X[train_idx], perm_labels[train_idx])
                preds = lda.predict(X[test_idx])
                correct += np.sum(preds == perm_labels[test_idx])
                total += len(test_idx)
            null_dist[p, w] = correct / total

    # p-value: proportion of null accuracies >= observed accuracy
    p_values = np.mean(null_dist >= acc, axis=0)

    return acc, p_values, null_dist


# build feature matrices

# species decoding 
# each observation: (dog, species), ERP averaged across valence and electrodes
species_data = (
    df.groupby(['ID', 'species'])[TIME_WINDOWS]
    .mean()
    .reset_index()
)
species_feature_matrix = species_data[TIME_WINDOWS].values
species_labels = (species_data['species'] == 'human').astype(int).values
species_groups = species_data['ID'].astype(int).values

# valence decoding
# each observation: (dog, valence), ERP averaged across species and electrodes
valence_data = (
    df.groupby(['ID', 'valence'])[TIME_WINDOWS]
    .mean()
    .reset_index()
)
valence_feature_matrix = valence_data[TIME_WINDOWS].values
valence_labels = (valence_data['valence'] == 'positive').astype(int).values
valence_groups = valence_data['ID'].astype(int).values

# run decoding

print("Running species decoding...")
species_acc, species_p, species_null = run_loo_lda(
    species_feature_matrix, species_labels, species_groups
)

print("Running valence decoding...")
valence_acc, valence_p, valence_null = run_loo_lda(
    valence_feature_matrix, valence_labels, valence_groups
)

# FDR correction (Benjamini-Hochberg) across windows

def fdr_bh(p_values, alpha=ALPHA):
    """Returns boolean array of which windows survive FDR correction."""
    n = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    threshold = (np.arange(1, n + 1) / n) * alpha
    below = sorted_p <= threshold
    if not np.any(below):
        return np.zeros(n, dtype=bool)
    max_sig = np.max(np.where(below)[0])
    sig = np.zeros(n, dtype=bool)
    sig[sorted_idx[:max_sig + 1]] = True
    return sig

species_sig = fdr_bh(species_p)
valence_sig = fdr_bh(valence_p)

# print results
print("\n" + "="*60)
print("SPECIES DECODING (human vs. dog)")
print("="*60)
print(f"{'Window':<12} {'Accuracy':>10} {'p-value':>10} {'FDR sig':>10}")
print("-"*45)
for w, window in enumerate(TIME_WINDOWS):
    sig_str = "*" if species_sig[w] else ""
    print(f"{window:<12} {species_acc[w]:>10.3f} {species_p[w]:>10.3f} {sig_str:>10}")

print(f"\nPeak accuracy: {species_acc.max():.3f} at window "
      f"{TIME_WINDOWS[species_acc.argmax()]} ms")
sig_windows = [TIME_WINDOWS[w] for w in range(N_WINDOWS) if species_sig[w]]
print(f"Windows surviving FDR correction: "
      f"{sig_windows if sig_windows else 'none'}")

print("\n" + "="*60)
print("VALENCE DECODING (positive vs. neutral)")
print("="*60)
print(f"{'Window':<12} {'Accuracy':>10} {'p-value':>10} {'FDR sig':>10}")
print("-"*45)
for w, window in enumerate(TIME_WINDOWS):
    sig_str = "*" if valence_sig[w] else ""
    print(f"{window:<12} {valence_acc[w]:>10.3f} {valence_p[w]:>10.3f} {sig_str:>10}")

print(f"\nPeak accuracy: {valence_acc.max():.3f} at window "
      f"{TIME_WINDOWS[valence_acc.argmax()]} ms")
sig_windows_v = [TIME_WINDOWS[w] for w in range(N_WINDOWS) if valence_sig[w]]
print(f"Windows surviving FDR correction: "
      f"{sig_windows_v if sig_windows_v else 'none'}")

# plot
window_centers = [int(w.split('-')[0]) + 25 for w in TIME_WINDOWS]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(window_centers, species_acc, color='steelblue', linewidth=2,
        label='Species decoding (human vs. dog)')
ax.plot(window_centers, valence_acc, color='firebrick', linewidth=2,
        linestyle='--', label='Valence decoding (positive vs. neutral)')

# Mark FDR-significant windows
for w in range(N_WINDOWS):
    if species_sig[w]:
        ax.axvspan(window_centers[w] - 25, window_centers[w] + 25,
                   alpha=0.15, color='steelblue')
    if valence_sig[w]:
        ax.axvspan(window_centers[w] - 25, window_centers[w] + 25,
                   alpha=0.15, color='firebrick')

ax.axhline(0.5, color='black', linewidth=1, linestyle=':', label='Chance (50%)')
ax.set_xlabel('Time after stimulus onset (ms)', fontsize=12)
ax.set_ylabel('LOO Decoding Accuracy', fontsize=12)
ax.set_title('Leave-One-Dog-Out LDA Decoding Accuracy', fontsize=13)
ax.set_ylim(0.2, 1.05)
ax.set_xlim(0, 1000)
ax.legend(fontsize=11)
ax.text(0.98, 0.05,
        'Shaded = FDR-corrected significant windows (α = 0.05)',
        transform=ax.transAxes, fontsize=9, ha='right', color='gray')

plt.tight_layout()
plt.savefig('result4_decoding.png', dpi=150,
            bbox_inches='tight')
print("\nFigure saved")
plt.close()

Running species decoding...
Running valence decoding...

SPECIES DECODING (human vs. dog)
Window         Accuracy    p-value    FDR sig
---------------------------------------------
0-100             0.529      0.568           
50-150            0.529      0.607           
100-200           0.559      0.521           
150-250           0.529      0.593           
200-300           0.471      0.749           
250-350           0.529      0.593           
300-400           0.618      0.233           
350-450           0.647      0.099           
400-500           0.647      0.062           
450-550           0.618      0.127           
500-600           0.618      0.216           
550-650           0.588      0.404           
600-700           0.588      0.374           
650-750           0.500      0.686           
700-800           0.324      1.000           
750-850           0.324      1.000           
800-900           0.471      0.708           
850-950           0.412      0.822  